# Chapter 12: Prompt Engineering & In-Context Learning
## Notebook 02 — Advanced Prompting

This notebook covers reasoning-heavy prompt patterns: **chain-of-thought (CoT)**, **self-consistency**, **ReAct** (reason + act with tools), **function/tool calling**, **JSON-mode parsing**, and a preview of **retrieval cues** that lead into Chapter 13 (RAG). We close with the **limits** of prompting (hallucination, context window, instruction following).

### What you'll learn

| Topic | Section |
|-------|--------|
| Chain-of-thought prompting | §1 |
| Self-consistency (sample + majority vote) | §2 |
| ReAct (reason + act) | §3 |
| Tool / function calling and JSON-mode parsing | §4 |
| Retrieval cues (preview of RAG) | §5 |
| Prompt patterns: persona, role, format, constraints, examples | §6 |
| Limits: hallucination, context window, instruction following | §7 |

**Estimated time:** 1.5–2 hours

---
*Generated by Berta AI*

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

from prompt_templates import (
    PromptTemplate, ChainOfThoughtTemplate, ReActTemplate,
    FewShotTemplate, FewShotExample,
)
from llm_clients import MockLLMClient
from evaluation_utils import safe_json_parse

import json
from collections import Counter

client = MockLLMClient(model='mock-llm-v1', temperature=0.0)
print('Setup complete.')

---
## 1. Chain-of-Thought (CoT)

**CoT** asks the model to **show its reasoning** before the final answer. Two flavors:

- **Zero-shot CoT** — append "Let's think step by step." (Kojima et al., 2022)
- **Few-shot CoT** — provide example problems with worked solutions (Wei et al., 2022)

Why it works: writing intermediate steps gives the model more **compute-per-question** in token form, and reduces shortcut errors on multi-step tasks.

In [ ]:
# Zero-shot CoT vs direct answer
question = 'A bag has 3 red marbles, 2 blue marbles, and 4 green marbles. How many marbles total?'

direct = PromptTemplate(name='direct', template='Q: {{ question }}\nA:')
cot = ChainOfThoughtTemplate(name='cot', template='')

print('--- Direct ---')
print(client.complete(direct.render(question=question)).text)
print('\n--- CoT ---')
print(client.complete(cot.render(input=question)).text)

---
## 2. Self-Consistency

A single CoT chain can be wrong. **Self-consistency** (Wang et al., 2022) samples **multiple** reasoning chains at temperature > 0 and takes the **majority vote** over the final answers.

We simulate this with the mock client (which honors `temperature` and `seed` for reproducible variation).

In [ ]:
def extract_answer(text):
    # Pull the last integer in the response.
    import re
    m = re.findall(r'-?\d+', text)
    return int(m[-1]) if m else None

samples = []
question = 'A bag has 3 red, 2 blue, and 4 green marbles. How many marbles total?'
prompt = cot.render(input=question)

for seed in range(7):
    text = client.complete(prompt, temperature=0.7, seed=seed).text
    ans = extract_answer(text)
    samples.append(ans)
    print(f'seed={seed}: ans={ans}')

vote = Counter(samples).most_common(1)[0]
print(f'\nMajority vote: {vote[0]}  (count={vote[1]}/{len(samples)})')

---
## 3. ReAct: Reason + Act

**ReAct** (Yao et al., 2022) interleaves **Thought** lines (reasoning) with **Action** lines (tool calls). The runtime executes each Action and appends an **Observation**. The loop terminates when the model emits `Action: Finish[answer]`.

This is the foundation of modern agents: the LLM decides **what to do next**, not just **what to say**.

In [ ]:
# Mock tools
def calculator(expr: str) -> str:
    try:
        return str(eval(expr, {'__builtins__': {}}, {}))
    except Exception as e:
        return f'ERROR: {e}'

def search(query: str) -> str:
    facts = {
        'capital of france': 'Paris',
        'speed of light': '299792458 m/s',
    }
    return facts.get(query.lower(), 'No result')

TOOLS = {'Calculator': calculator, 'Search': search}

react = ReActTemplate(
    name='react_v1',
    template='',
    tools_description='Calculator[<expr>] -- evaluate a math expression\nSearch[<query>] -- look up a fact\nFinish[<answer>] -- stop and return the answer',
)
print(react.render(input='What is 17 + 25?')[:300] + '...')

In [ ]:
def run_react(question, max_steps=4):
    import re
    scratchpad = ''
    for step in range(max_steps):
        prompt = react.render(input=question, scratchpad=scratchpad)
        out = client.complete(prompt).text.strip()
        # Parse the model output: it should contain Thought / Action lines.
        match = re.search(r'(.*?)Action:\s*(\w+)\[(.*?)\]', out, re.DOTALL)
        if not match:
            scratchpad += '\nThought:' + out
            break
        thought, tool, arg = match.group(1).strip(), match.group(2), match.group(3)
        print(f'Step {step+1}: tool={tool}({arg!r})')
        if tool == 'Finish':
            scratchpad += f'\nThought:{thought}\nAction: Finish[{arg}]'
            return arg, scratchpad
        result = TOOLS.get(tool, lambda x: 'unknown tool')(arg)
        scratchpad += f'\nThought:{thought}\nAction: {tool}[{arg}]\nObservation: {result}'
    return None, scratchpad

answer, trace = run_react('What is 17 + 25?')
print('\nFinal answer:', answer)
print('\n--- Scratchpad ---')
print(trace)

---
## 4. Tool / Function Calling and JSON-Mode Parsing

When you want the model to **invoke a function** with structured arguments, ask for **JSON conforming to a tool schema**. Modern providers expose a native "tool calling" mode; underneath, it's still prompt + JSON parsing — which we can build by hand.

In [ ]:
TOOL_SCHEMA = {
    'name': 'lookup_weather',
    'description': 'Get current weather for a city',
    'parameters': {
        'type': 'object',
        'properties': {
            'city': {'type': 'string'},
            'units': {'type': 'string', 'enum': ['c', 'f']},
        },
        'required': ['city'],
    },
}

prompt = (
    'Available tool:\n' + json.dumps(TOOL_SCHEMA, indent=2) +
    '\n\nUser: What is the weather in Berlin in Celsius?'
    '\nReturn JSON: {"tool": "lookup_weather", "args": {...}}'
)

raw = client.complete(prompt).text
print('Raw response:', raw)
parsed = safe_json_parse(raw)
print('Parsed:', parsed)
print('Tool to call:', (parsed or {}).get('tool', 'NONE'))

**JSON-mode tip.** When parsing fails, **don't crash** — log the raw output, return a graceful fallback ("I couldn't determine that"), and surface a metric so you can iterate on the prompt.

---
## 5. Retrieval Cues (Preview of RAG)

When the model lacks information, **inject retrieved context** into the prompt. The pattern looks like:

```
Context (from search):
- <doc 1>
- <doc 2>

Question: ...
Answer using only the context above.
```

This is the foundation of **Retrieval-Augmented Generation (RAG)**, covered in **Chapter 13**.

In [ ]:
rag = PromptTemplate(
    name='rag_preview',
    system='Answer using only the provided context. If the context does not answer the question, say "I do not know."',
    template='Context:\n{{ context }}\n\nQuestion: {{ question }}\nAnswer:',
)

context = '- Paris is the capital of France.\n- France is a country in Western Europe.'
print(rag.render(context=context, question='What is the capital of France?'))
print('\nResponse:', client.complete(rag.render(context=context, question='What is the capital of France?')).text)

---
## 6. Prompt Patterns

A handful of recurring patterns cover most production needs. Use these as Lego blocks.

| Pattern | When to use | Example cue |
|---------|-------------|-------------|
| **Persona** | Voice / domain expertise | "You are a senior auditor." |
| **Role** | Task framing | "Act as a JSON-only classifier." |
| **Format** | Output discipline | "Return one of: yes, no, unknown." |
| **Constraints** | Length, language, safety | "Answer in two sentences." |
| **Examples (few-shot)** | Narrow / unusual tasks | k=3–5 worked examples |

In [ ]:
# Compose multiple patterns into one prompt
composed = PromptTemplate(
    name='composed_v1',
    system='You are a senior tech writer (persona). Act as a one-line summarizer (role).',
    template=(
        'Constraints: respond in ONE sentence, under 20 words.\n'
        'Format: plain text only — no markdown.\n\n'
        'Examples:\n'
        '- Input: A long article about LLM costs. -> Output: LLM costs are dominated by output tokens.\n'
        '- Input: A study about sleep and learning. -> Output: Sleep boosts memory consolidation.\n\n'
        'Input: {{ text }}\nOutput:'
    ),
)
print(composed.render(text='Researchers announced a new battery design that promises faster charging.'))

---
## 7. Limits of Prompting

- **Hallucination.** Models invent confident wrong answers, especially on facts. Mitigations: ground in **retrieved context** (Ch 13), require **citations**, lower temperature, validate against a tool.
- **Context window.** Inputs above the limit are truncated silently. Mitigations: chunking, summarization, retrieval.
- **Instruction following.** Long or contradictory instructions degrade quality. Keep prompts **short and ordered**: most-important rules first.
- **Sensitivity to wording.** A 3-word change can flip the answer. Mitigations: A/B test (Notebook 03), pin a canonical version.
- **Non-determinism.** Even at temperature 0, providers may not be bit-stable. Mitigations: set seeds when offered, retry idempotently.

---
## 8. Key Takeaways

- **CoT** trades tokens for accuracy on reasoning tasks.
- **Self-consistency** (sample + vote) further boosts CoT.
- **ReAct** turns the LLM into an agent that calls tools.
- **Tool calling = JSON parsing**: pair every JSON prompt with `safe_json_parse` and a schema.
- Compose **persona + role + format + constraints + examples** as needed; don't over-stuff.
- Know the **limits**: hallucination, context window, wording sensitivity.

Next: **Notebook 03** — evaluation harnesses, A/B testing, prompt-injection defenses, and production wiring.

---
*Generated by Berta AI*